# Flat 4 / 5 / 7-class 모델 비교

동일한 검증 타일에서 세 모델의 **원래 클래스 지표**, **공통 4-class 지표**, **GT/예측 이미지**를 비교합니다.
학습은 수행하지 않으며 기본값은 각 실험의 `best_model.pt`입니다.

- 학습 노트북의 slide-balanced split을 다시 계산합니다. 데이터가 바뀌면 split도 달라질 수 있습니다.
- 이 데이터는 모델 선택에 사용된 validation이며 독립적인 test 성능은 아닙니다.
- 원래 클래스별 Macro F1은 클래스 정의가 달라 직접 순위를 매기지 않습니다.
- 공통 평가: Negative Epithelial / Positive Epithelial / Positive Immune / other.
  Negative Immune은 other에 합칩니다. Flat sigmoid 출력은 합산하지 않고 같은 그룹의 **최대 score**로 합칩니다.
  클래스 무관 NMS를 사용하므로 원래 최고 점수와 위치 검출을 유지하는 방식입니다(동점은 예외).
- 라벨 품질을 고려해 수치와 동일 타일의 육안 비교를 함께 확인하세요. Confidence는 모델 간 보정되어 있지 않습니다.

위에서부터 실행하세요. `GPU_ID`, 체크포인트 경로, `VIS_INDICES`, `CONFIDENCE_THRESHOLD`를 첫 코드 셀에서 바꿀 수 있습니다.

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import gc
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Patch
from IPython.display import display
import torch
from scipy.optimize import linear_sum_assignment  # 평가 의존성 사전 확인
from nets import nn
from utils import util
from utils.ihc_dataset import (
    IHCHierarchicalDataset, PDL1_CLASS_NAMES, PDL1_CLASS_COLORS,
    discover_records, split_records_by_slide_balanced, slide_id_from_record,
)
from utils.valid import compute_point_label_metrics_single

GPU_ID = 0
DATA_ROOT = Path('../../data/mediautoAI_PDL1_Stomach')
MODEL_ROOT = Path('../../model/mediautoAI_PDL1_Stomach')
CHECKPOINT_NAME = 'last_model.pt'  # 'best_model.pt'로 변경 가능
INPUT_SIZE, RESIZE_SIZE = 512, 1024
VALIDATION_SLIDE_COUNT = 3
BATCH_SIZE = 4
NUM_WORKERS = 0
CONFIDENCE_THRESHOLD = 0.10
NMS_IOU_THRESHOLD = 0.45  # 기존 평가 함수에 고정된 값과 동일
DISTANCE_THRESHOLD = 16
VIS_INDICES = None  # 예: [0, 4, 8]; None이면 검증 타일 중 6개를 고정 선택
SHOW_FIGURES = True
RUN_TAG = datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%d_%H%M%S_%f')
OUTPUT_DIR = Path('results') / f'flat_4_5_7_comparison_{RUN_TAG}'

COMMON_NAMES = {0: 'Negative Epithelial', 1: 'Positive Epithelial', 2: 'Positive Immune', 3: 'other'}
COMMON_COLORS = {0: '#1f77b4', 1: '#d62728', 2: '#ff7f0e', 3: '#7f7f7f'}
SOURCE_TO_COMMON = (0, 3, 3, 1, 2, 2, 3)
EXPERIMENTS = {
    'flat4': {
        'path': MODEL_ROOT / 'pdl1_flat_4class_negative_immune_other' / CHECKPOINT_NAME,
        'architecture': 'pdl1_flat_4class_negative_immune_other',
        'names': COMMON_NAMES, 'colors': COMMON_COLORS,
        'source_mapping': SOURCE_TO_COMMON, 'common_mapping': (0, 1, 2, 3),
    },
    'flat5': {
        'path': MODEL_ROOT / 'pdl1_flat_5class' / CHECKPOINT_NAME,
        'architecture': 'pdl1_flat_5class',
        'names': {0: 'Negative Epithelial', 1: 'Negative Immune', 2: 'Positive Epithelial', 3: 'Positive Immune', 4: 'other'},
        'colors': {0: '#1f77b4', 1: '#2ca02c', 2: '#d62728', 3: '#ff7f0e', 4: '#7f7f7f'},
        'source_mapping': (0, 1, 1, 2, 3, 3, 4), 'common_mapping': (0, 3, 1, 2, 3),
    },
    'flat7': {
        'path': MODEL_ROOT / 'pdl1_flat_7class' / CHECKPOINT_NAME,
        'architecture': 'pdl1_flat_7class',
        'names': PDL1_CLASS_NAMES.copy(), 'colors': PDL1_CLASS_COLORS.copy(),
        'source_mapping': tuple(range(7)), 'common_mapping': SOURCE_TO_COMMON,
    },
}
if torch.cuda.is_available():
    if not 0 <= GPU_ID < torch.cuda.device_count():
        raise ValueError(f'GPU_ID={GPU_ID}, available={torch.cuda.device_count()}')
    torch.cuda.set_device(GPU_ID)
    device = torch.device(f'cuda:{GPU_ID}')
else:
    device = torch.device('cpu')
missing = [str(spec['path']) for spec in EXPERIMENTS.values() if not spec['path'].is_file()]
if missing:
    raise FileNotFoundError('Missing checkpoints: ' + ', '.join(missing))
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
print(f'Device: {device} | Output: {OUTPUT_DIR.resolve()}')


In [ ]:
class RemappedDataset(torch.utils.data.Dataset):
    def __init__(self, base, mapping):
        self.base, self.mapping = base, tuple(mapping)
    def __len__(self):
        return len(self.base)
    def __getitem__(self, index):
        image, labels, boxes, indices = self.base[index]
        return image, labels.new_tensor(self.mapping)[labels.long()], boxes, indices

class CommonFourClassModel(torch.nn.Module):
    def __init__(self, model, mapping):
        super().__init__()
        self.model, self.mapping = model, tuple(mapping)
    def forward(self, images):
        raw = self.model(images)
        scores = raw[:, 4:]
        merged = torch.stack([
            scores[:, [i for i, group in enumerate(self.mapping) if group == cls]].amax(dim=1)
            for cls in range(4)
        ], dim=1)
        return torch.cat((raw[:, :4], merged), dim=1)

def make_loader(base, mapping):
    return torch.utils.data.DataLoader(
        RemappedDataset(base, mapping), batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, collate_fn=IHCHierarchicalDataset.collate_fn,
        pin_memory=(device.type == 'cuda'),
    )

def load_model(spec):
    # 로컬 학습 코드가 생성한 신뢰할 수 있는 체크포인트에 사용합니다.
    ckpt = torch.load(spec['path'], map_location='cpu', weights_only=False)
    if ckpt.get('num_classes') != len(spec['names']):
        raise ValueError(f"Class count mismatch: {spec['path']}")
    if ckpt.get('architecture') != spec['architecture']:
        raise ValueError(f"Architecture mismatch: {spec['path']}")
    saved_names = ckpt.get('class_names', ckpt.get('params', {}).get('names'))
    if saved_names is not None and {int(k): v for k, v in saved_names.items()} != spec['names']:
        raise ValueError(f"Class order mismatch: {spec['path']}")
    saved_mapping = ckpt.get('source_to_flat')
    if saved_mapping is not None and tuple(saved_mapping) != spec['source_mapping']:
        raise ValueError(f"Label mapping mismatch: {spec['path']}")
    model = nn.yolo_v11_m(num_classes=len(spec['names']))
    model.load_state_dict(ckpt['model_state_dict'], strict=True)
    metadata = {
        'checkpoint': str(spec['path'].resolve()), 'epoch': int(ckpt['epoch']) + 1,
        'saved_macro_f1': float(ckpt.get('macro_f1', float('nan'))),
        'cls_pos_weight': ckpt.get('params', {}).get('cls_pos_weight'),
    }
    del ckpt
    return model.to(device).eval(), metadata

def evaluate(model, loader, names):
    metrics = compute_point_label_metrics_single(
        model, loader, device, {'names': names}, distance_threshold=DISTANCE_THRESHOLD,
        class_agnostic_nms=True, confidence_threshold=CONFIDENCE_THRESHOLD,
    )
    if not metrics or 'class_stats' not in metrics:
        raise RuntimeError('Evaluation returned no metrics')
    return metrics


In [ ]:
records = discover_records(DATA_ROOT)
train_records, val_records = split_records_by_slide_balanced(records, val_slide_count=VALIDATION_SLIDE_COUNT)
train_slides = {slide_id_from_record(r) for r in train_records}
val_slides = {slide_id_from_record(r) for r in val_records}
assert train_slides.isdisjoint(val_slides)
base_dataset = IHCHierarchicalDataset(
    val_records, input_size=INPUT_SIZE, resize_size=RESIZE_SIZE,
    augment=False, random_crop=False, cache_images=True, four_crop=True,
)
if VIS_INDICES is None:
    VIS_INDICES = np.linspace(0, len(base_dataset)-1, min(6, len(base_dataset)), dtype=int).tolist()
if not VIS_INDICES or any(not 0 <= i < len(base_dataset) for i in VIS_INDICES):
    raise ValueError('VIS_INDICES must contain valid validation tile indices')
manifest = pd.DataFrame([
    {'tile_index': i, 'image': str(val_records[i//4][0]),
     'label': str(val_records[i//4][1]), 'quadrant_index': i % 4,
     'visualized': i in VIS_INDICES}
    for i in range(len(base_dataset))
])
manifest.to_csv(OUTPUT_DIR / 'validation_tiles.csv', index=False)
print(f'Validation: {len(val_slides)} slides, {len(val_records)} patches, {len(base_dataset)} tiles')
print('\n'.join(sorted(val_slides)))
display(manifest[manifest.visualized])


## 같은 타일에서 수치 평가 및 예측 수집
모델을 하나씩 GPU에 올려 평가합니다. 원래 클래스 및 공통 4-class 평가 모두 동일한 confidence=0.10(설정값), class-agnostic NMS IoU=0.45, 중심점 거리 기준을 사용합니다.
검출 Recall은 클래스 일치와 무관하며, 분류 정확도는 위치가 매칭된 세포에 한정됩니다.

In [ ]:
summary_rows, class_rows, checkpoint_rows = [], [], []
visual_predictions = {}
for model_name, spec in EXPERIMENTS.items():
    print(f'Evaluating {model_name} ...')
    model, metadata = load_model(spec)
    checkpoint_rows.append({'model': model_name, **metadata})
    common_model = CommonFourClassModel(model, spec['common_mapping']).eval()
    for scope, eval_model, mapping, names in [
        ('native', model, spec['source_mapping'], spec['names']),
        ('common4', common_model, SOURCE_TO_COMMON, COMMON_NAMES),
    ]:
        metrics = evaluate(eval_model, make_loader(base_dataset, mapping), names)
        summary_rows.append({
            'model': model_name, 'scope': scope, 'epoch': metadata['epoch'],
            **{key: float(metrics[key]) for key in (
                'detection_recall', 'classification_accuracy', 'macro_precision',
                'macro_recall', 'macro_f1', 'overall_recall')},
        })
        for name, stats in metrics['class_stats'].items():
            class_rows.append({'model': model_name, 'scope': scope, 'class': name, **stats})
    visual_predictions[model_name] = {}
    with torch.inference_mode():
        for index in VIS_INDICES:
            images = base_dataset[index][0].unsqueeze(0).to(device).float() / 255
            # 평가와 동일한 mixed precision 설정
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                predictions = model(images)
            native = util.non_max_suppression(
                predictions, confidence_threshold=CONFIDENCE_THRESHOLD,
                iou_threshold=NMS_IOU_THRESHOLD, class_agnostic=True,
            )[0].detach().cpu()
            common = native.clone()
            if len(common):
                common[:, 5] = common.new_tensor(spec['common_mapping'])[native[:, 5].long()]
            visual_predictions[model_name][index] = {'native': native, 'common4': common}
    del common_model, model
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()

summary = pd.DataFrame(summary_rows)
per_class = pd.DataFrame(class_rows)
checkpoints = pd.DataFrame(checkpoint_rows)
summary.to_csv(OUTPUT_DIR / 'summary.csv', index=False)
per_class.to_csv(OUTPUT_DIR / 'per_class.csv', index=False)
checkpoints.to_csv(OUTPUT_DIR / 'checkpoints.csv', index=False)
settings = {
    'confidence': CONFIDENCE_THRESHOLD, 'nms_iou': NMS_IOU_THRESHOLD,
    'distance_pixels': DISTANCE_THRESHOLD, 'input_size': INPUT_SIZE,
    'resize_size': RESIZE_SIZE, 'validation_slides': sorted(val_slides),
    'visual_indices': VIS_INDICES, 'checkpoint_name': CHECKPOINT_NAME,
    'experiments': EXPERIMENTS,
}
(OUTPUT_DIR / 'settings.json').write_text(json.dumps(settings, ensure_ascii=False, indent=2, default=str))
print('Native metrics (클래스 정의가 서로 다름)')
display(summary[summary.scope == 'native'].round(4))
print('Common 4-class metrics (동일 클래스 정의)')
display(summary[summary.scope == 'common4'].round(4))
display(checkpoints)


In [ ]:
common_summary = summary[summary.scope == 'common4'].set_index('model')
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
common_summary[['detection_recall', 'macro_f1', 'classification_accuracy']].plot.bar(ax=axes[0], ylim=(0, 1), rot=0)
axes[0].set_title('Common 4-class evaluation')
class_f1 = per_class[per_class.scope == 'common4'].pivot(index='class', columns='model', values='f1').reindex(COMMON_NAMES.values())
class_f1.plot.bar(ax=axes[1], ylim=(0, 1), rot=20)
axes[1].set_title('Common per-class F1')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'metrics_comparison.png', dpi=160)
plt.show()
display(per_class.round(4))


## 육안 비교: 공통 클래스와 각 모델의 원래 클래스
각 타일에 대해 공통 색상의 **GT / flat4 / flat5 / flat7** 한 줄 비교와,
모델별 원래 클래스의 **GT / 예측** 3행 비교를 저장합니다.
GT는 제공된 라벨이며 미표기 세포가 있을 수 있습니다. 모델 간 색상과 위치, 중복 검출 및 누락을 함께 확인하세요.

In [ ]:
def draw_panel(axis, image, boxes, labels, names, colors, title, scores=None):
    axis.imshow(image)
    for j, (box, label) in enumerate(zip(boxes, labels)):
        cls = int(label)
        x1, y1, x2, y2 = [float(v) for v in box]
        axis.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor=colors[cls], linewidth=0.7))
    axis.set_title(f'{title} | n={len(labels)}', fontsize=10)
    axis.axis('off')
    axis.legend(handles=[Patch(color=colors[c], label=f'{names[c]}: {int((labels == c).sum())}') for c in names],
                loc='upper left', fontsize=6, framealpha=.8)

def ground_truth(index, mapping):
    image, labels, boxes, _ = base_dataset[index]
    labels = labels.new_tensor(mapping)[labels.long()]
    xywh = boxes * INPUT_SIZE
    xyxy = torch.cat((xywh[:, :2]-xywh[:, 2:]/2, xywh[:, :2]+xywh[:, 2:]/2), dim=1)
    return image.permute(1, 2, 0).numpy(), labels, xyxy

for index in VIS_INDICES:
    image, labels, boxes = ground_truth(index, SOURCE_TO_COMMON)
    fig, axes = plt.subplots(1, 4, figsize=(24, 6))
    draw_panel(axes[0], image, boxes, labels, COMMON_NAMES, COMMON_COLORS, 'GT (common4)')
    for axis, name in zip(axes[1:], EXPERIMENTS):
        pred = visual_predictions[name][index]['common4']
        draw_panel(axis, image, pred[:, :4], pred[:, 5], COMMON_NAMES, COMMON_COLORS, name)
    fig.suptitle(f'Tile {index} | {Path(val_records[index//4][0]).parent.name} / {Path(val_records[index//4][0]).name} | quadrant={index%4}', fontsize=9)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'tile_{index:03d}_common4.png', dpi=150)
    if SHOW_FIGURES: plt.show()
    plt.close(fig)

    fig, axes = plt.subplots(3, 2, figsize=(14, 20))
    for row, (name, spec) in enumerate(EXPERIMENTS.items()):
        image, labels, boxes = ground_truth(index, spec['source_mapping'])
        draw_panel(axes[row, 0], image, boxes, labels, spec['names'], spec['colors'], f'{name} GT')
        pred = visual_predictions[name][index]['native']
        draw_panel(axes[row, 1], image, pred[:, :4], pred[:, 5], spec['names'], spec['colors'], f'{name} prediction')
    fig.suptitle(f'Tile {index}: native classes')
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'tile_{index:03d}_native.png', dpi=150)
    if SHOW_FIGURES: plt.show()
    plt.close(fig)
print(f'Results saved: {OUTPUT_DIR.resolve()}')
